In [7]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import math
import pickle

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# from fix_data_leakage import scale_data_without_leakage, create_sequences
from sklearn.preprocessing import StandardScaler

In [8]:
!git clone https://github.com/HoangHumg1210/hoankiem-air-quality-.git
%cd hoankiem-air-quality-


Cloning into 'hoankiem-air-quality-'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 32 (delta 3), reused 32 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 6.94 MiB | 13.88 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/hoankiem-air-quality-/hoankiem-air-quality-


In [9]:
df = pd.read_csv('data/processed/data2225_clean.csv')
df.head()

,index,CO,NO2,O3,PM10,PM25,SO2,Clouds,Precipitation,Pressure,Relative Humidity,Temperature,UV Index,Wind Speed,HolidayName,IsHoliday,Accumulated Hours of Rain
0,0,353.1,10.0,84.0,98.0,17.08,52.0,100,0.00,1020.0,95.0,15.5,0.6,2.00,Ngày thường,False,0
1,1,343.5,9.0,87.3,95.7,16.75,48.7,91,0.00,1021.0,94.0,15.4,0.7,2.33,Ngày thường,False,0
2,2,334.0,8.0,90.7,93.3,16.42,45.3,83,0.50,1022.0,93.0,15.3,1.0,2.66,Ngày thường,False,1
3,3,324.5,7.0,94.0,91.0,16.09,42.0,75,0.75,1022.0,93.0,15.2,1.5,3.00,Ngày thường,False,2
4,4,319.6,6.7,95.7,91.3,16.17,39.0,83,0.00,1021.0,87.0,15.6,1.9,3.00,Ngày thường,False,0


In [10]:
df.drop(columns=['index'], inplace=True)
df['Local Time'] = pd.to_datetime(df['Local Time'])
df = df.set_index('Local Time').sort_index()

KeyError: 'Local Time'

In [ ]:
df.info()

In [ ]:
dup = df.index.duplicated().sum()
print("Duplicate timestamps:", dup)
print("Min/Max:", df.index.min(), df.index.max())

df = df[~df.index.duplicated(keep="last")]
print("Duplicate timestamps:", df.index.duplicated().sum())
# if dup>0:
#     df = df.groupby(df.index).mean(numeric_only=True)

# print('Check', dup)

In [ ]:
# 2) ép về tần suất 1 giờ để lộ missing timestamp (KHÔNG lấy mean gộp 3h)
df = df.asfreq("1H")

# 3) xử lý missing trên dữ liệu gốc
# gợi ý: nội suy theo thời gian cho numeric
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].interpolate(method="time").ffill().bfill()

# IsHoliday nếu có: forward fill (thường là nhãn theo ngày)
if "IsHoliday" in df.columns:
    df["IsHoliday"] = df["IsHoliday"].ffill().bfill().astype(int)

In [ ]:
idx = df.index
df["day_of_week"] = idx.dayofweek
df["is_weekend"] = (idx.dayofweek >= 5).astype(int)

# hour cyclical
df["hour_sin"] = np.sin(2*np.pi*idx.hour/24)
df["hour_cos"] = np.cos(2*np.pi*idx.hour/24)

# month cyclical (đừng để month kiểu int đơn thuần)
df["month_sin"] = np.sin(2*np.pi*idx.month/12)
df["month_cos"] = np.cos(2*np.pi*idx.month/12)

In [ ]:
target = "PM25"

lags = [1, 3, 6, 12, 24, 48]  # 1h, 3h, 6h, 12h, 1d, 2d (tuỳ bạn)
for l in lags:
    df[f"{target}_lag{l}"] = df[target].shift(l)

# rolling theo giờ
df[f"{target}_roll_mean_24"] = df[target].rolling(24).mean()
df[f"{target}_roll_std_24"]  = df[target].rolling(24).std()
df[f"{target}_roll_mean_72"] = df[target].rolling(72).mean()

df = df.dropna()

In [ ]:
train_end = "2023-12-31"
val_start = "2024-01-01"
val_end = "2024-12-31"
test_start = "2025-01-01"

train_df = df[:train_end]
val_df = df[val_start:val_end]
test_df = df[test_start:]

print(train_df.shape, val_df.shape, test_df.shape)


In [ ]:
USE_LOG_TARGET = True

# y raw
y_train_raw = train_df[[target]].values
y_val_raw   = val_df[[target]].values
y_test_raw  = test_df[[target]].values

def forward_target_transform(y):
    y = np.asarray(y, dtype=np.float64)
    return np.log1p(np.clip(y, 0, None)) if USE_LOG_TARGET else y

def inverse_target_transform(y):
    y = np.asarray(y, dtype=np.float64)
    return np.expm1(y) if USE_LOG_TARGET else y


y_train_t = forward_target_transform(y_train_raw)
y_val_t   = forward_target_transform(y_val_raw)
y_test_t  = forward_target_transform(y_test_raw)



scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train_t)
y_val   = scaler_y.transform(y_val_t)
y_test  = scaler_y.transform(y_test_t)

def create_sequences(X, y, lookback=72, horizon=1):
    y = np.asarray(y)
    if y.ndim == 1:
        y = y.reshape(-1, 1)

    X_seq, y_seq = [], []
    for i in range(lookback, len(X) - horizon + 1):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i + horizon - 1])
    return np.array(X_seq), np.array(y_seq)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)

# 1. Raw PM2.5
axes[0].plot(train_df.index, y_train_raw, label='Train', alpha=0.8)
axes[0].plot(val_df.index, y_val_raw, label='Val', alpha=0.8)
axes[0].plot(test_df.index, y_test_raw, label='Test', alpha=0.8)
axes[0].set_title('Bước 1: PM2.5 gốc (Raw)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Sau log1p transform
axes[1].plot(train_df.index, y_train_t, label='Train', alpha=0.8)
axes[1].plot(val_df.index, y_val_t, label='Val', alpha=0.8)
axes[1].plot(test_df.index, y_test_t, label='Test', alpha=0.8)
axes[1].set_title('Bước 2: Sau log1p transform', fontsize=13, fontweight='bold')
axes[1].set_ylabel('log1p(PM2.5)')
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. Sau StandardScaler (fit trên train)
axes[2].plot(train_df.index, y_train, label='Train', alpha=0.8)
axes[2].plot(val_df.index, y_val, label='Val', alpha=0.8)
axes[2].plot(test_df.index, y_test, label='Test', alpha=0.8)
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Mean (train)')
axes[2].set_title('Bước 3: Sau StandardScaler (fit trên Train)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Scaled value')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Histogram so sánh phân phối ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(y_train_raw.ravel(), bins=50, alpha=0.7, color='#4e79a7', edgecolor='white')
axes[0].set_title('Phân phối PM2.5 gốc (Train)')
axes[0].set_xlabel('PM2.5')

axes[1].hist(y_train_t.ravel(), bins=50, alpha=0.7, color='#f28e2b', edgecolor='white')
axes[1].set_title('Phân phối sau log1p (Train)')
axes[1].set_xlabel('log1p(PM2.5)')

axes[2].hist(y_train.ravel(), bins=50, alpha=0.7, color='#e15759', edgecolor='white')
axes[2].set_title('Phân phối sau StandardScaler (Train)')
axes[2].set_xlabel('Scaled value')

plt.tight_layout()
plt.show()

# --- Thống kê ---
print("=== Thống kê qua từng bước ===")
for name, data in [("Raw", y_train_raw), ("Log1p", y_train_t), ("Scaled", y_train)]:
    d = data.ravel()
    print(f"{name:8s} | Mean: {d.mean():8.3f} | Std: {d.std():7.3f} | "
          f"Min: {d.min():8.3f} | Max: {d.max():7.3f} | Skew: {pd.Series(d).skew():6.3f}")


In [ ]:
bad_cols = train_df.drop(columns=[target]).select_dtypes(include=["object"]).columns
print("Object columns:", bad_cols.tolist())
print(train_df[bad_cols].head())

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X_train_df = train_df.drop(columns=[target])
X_val_df   = val_df.drop(columns=[target])
X_test_df  = test_df.drop(columns=[target])

num_cols = X_train_df.select_dtypes(include=[np.number, "bool"]).columns
cat_cols = X_train_df.select_dtypes(include=["object"]).columns  # HolidayName

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

X_train = preprocess.fit_transform(X_train_df)
X_val   = preprocess.transform(X_val_df)
X_test  = preprocess.transform(X_test_df)

In [ ]:
LOOKBACK = 168   # 72 giờ (3 ngày) nhìn lại
HORIZON  = 24    # Dự đoán 1 bước tiếp theo

X_train_seq, y_train_seq = create_sequences(X_train, y_train, LOOKBACK, HORIZON)
X_val_seq,   y_val_seq   = create_sequences(X_val,   y_val,   LOOKBACK, HORIZON)
X_test_seq,  y_test_seq  = create_sequences(X_test,  y_test,  LOOKBACK, HORIZON)

print(X_train_seq.shape, y_train_seq.shape)
print(X_val_seq.shape,   y_val_seq.shape)
print(X_test_seq.shape,  y_test_seq.shape)


In [ ]:
n_features = X_train_seq.shape[2]  # Số features
model = Sequential([
    # CNN block — trích xuất local patterns
    Conv1D(filters=64, kernel_size=3, activation='relu', 
           padding='same', input_shape=(LOOKBACK, n_features)),
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    
    # GRU block — nắm bắt temporal dependencies
    GRU(128, return_sequences=True),
    Dropout(0.2),
    GRU(64,  return_sequences=False),
    Dropout(0.2),
    
    # Dense output
    Dense(32, activation='relu'),
    Dense(1)  # Dự đoán PM2.5
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True,
    verbose=1
)
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=100,
    batch_size=64,
    validation_data=(X_val_seq, y_val_seq),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Dự đoán
y_pred_scaled = model.predict(X_test_seq)

# Inverse transform (StandardScaler → log1p → giá trị gốc)
y_pred_t = scaler_y.inverse_transform(y_pred_scaled)
y_test_t_inv = scaler_y.inverse_transform(y_test_seq)

y_pred_raw = inverse_target_transform(y_pred_t)
y_test_raw_inv = inverse_target_transform(y_test_t_inv)

# Tính metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math

mae  = mean_absolute_error(y_test_raw_inv, y_pred_raw)
rmse = math.sqrt(mean_squared_error(y_test_raw_inv, y_pred_raw))

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")


In [ ]:
# Baseline 1: Naive — lấy giá trị PM2.5 giờ trước làm dự đoán
y_naive = y_test_raw_inv[:-1]  # PM2.5(t-1) dự đoán PM2.5(t)
y_true  = y_test_raw_inv[1:]
mae_naive  = mean_absolute_error(y_true, y_naive)
rmse_naive = math.sqrt(mean_squared_error(y_true, y_naive))
print(f"Naive Baseline — MAE: {mae_naive:.4f}, RMSE: {rmse_naive:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

HORIZON = 24

y_test_raw = test_df["PM25"].values  # PM25 thật

# true: từ t+24 trở đi
y_true = y_test_raw[HORIZON:]

# pred naive: lấy y_t để dự báo y_{t+24}
y_pred_naive = y_test_raw[:-HORIZON]

mae_naive = mean_absolute_error(y_true, y_pred_naive)
rmse_naive = np.sqrt(mean_squared_error(y_true, y_pred_naive))

print("Naive  MAE:", mae_naive)
print("Naive RMSE:", rmse_naive)

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(y_test_raw_inv[:500], label='Thực tế', alpha=0.8)
plt.plot(y_pred_raw[:500],     label='Dự đoán', alpha=0.8)
plt.legend()
plt.title('PM2.5 — Thực tế vs Dự đoán (500 mẫu đầu)')
plt.ylabel('PM2.5')
plt.show()
